# 🧹 Step-by-Step Data Cleaning Notebook

This notebook cleans the **Credit Risk Dataset** step-by-step with simple, clear pandas code.

### Step 1: Import Libraries

In [ ]:
import pandas as pd
import numpy as np

### Step 2: Load the Raw Dataset

In [ ]:
df = pd.read_csv('../data/row/credit_risk_dataset.csv')
print('Raw Data Shape:', df.shape)
df.head()

### Step 3: Rename Columns to Simple Names

In [ ]:
column_mapping = {
    'person_age': 'age',
    'person_income': 'income',
    'person_home_ownership': 'home_ownership',
    'person_emp_length': 'employment_years',
    'loan_intent': 'loan_purpose',
    'loan_grade': 'loan_grade',
    'loan_amnt': 'loan_amount',
    'loan_int_rate': 'interest_rate',
    'loan_status': 'default',
    'loan_percent_income': 'loan_income_ratio',
    'cb_person_default_on_file': 'previous_default',
    'cb_person_cred_hist_length': 'credit_history_years'
}

df = df.rename(columns=column_mapping)
df.head()

### Step 4: Check Column Types and Information

In [ ]:
df.info()

### Step 5: Check and Remove Duplicate Rows

In [ ]:
print('Duplicates found:', df.duplicated().sum())
df = df.drop_duplicates()
print('Shape after removing duplicates:', df.shape)

### Step 6: Check for Missing Values (Count)

In [ ]:
print(df.isnull().sum())

### Step 7: Check Missing Values Percentage

In [ ]:
missing_pct = (df.isnull().sum() / len(df)) * 100
print(missing_pct[missing_pct > 0].round(2))

### Step 8: Impute Missing Interest Rate by Loan Grade Median

In [ ]:
# Impute interest rate using median of each loan grade
df['interest_rate'] = df['interest_rate'].fillna(
    df.groupby('loan_grade')['interest_rate'].transform('median')
)
print('Missing interest_rate remaining:', df['interest_rate'].isnull().sum())

### Step 9: Impute Missing Employment Years with Median

In [ ]:
# Impute employment years with overall median
df['employment_years'] = df['employment_years'].fillna(df['employment_years'].median())
print('Missing employment_years remaining:', df['employment_years'].isnull().sum())

### Step 10: Verify All Missing Values Are Filled

In [ ]:
print('Total missing values in dataset:', df.isnull().sum().sum())

### Step 11: Check and Filter Age Outliers (18 to 100 Years)

In [ ]:
print('Records with age > 100:')
print(df[df['age'] > 100][['age', 'income', 'loan_amount']])

# Keep only valid ages between 18 and 100
df = df[(df['age'] >= 18) & (df['age'] <= 100)]
print('Age range after cleaning:', df['age'].min(), 'to', df['age'].max())

### Step 12: Check and Filter Employment Years Outliers (> 60 Years)

In [ ]:
print('Records with employment_years > 60:')
print(df[df['employment_years'] > 60][['age', 'income', 'employment_years']])

# Remove impossible values > 60 years
df = df[df['employment_years'] <= 60]
print('Max employment years after cleaning:', df['employment_years'].max())

### Step 13: Check That Employment Years Does Not Exceed Working Age

In [ ]:
# An applicant cannot work more years than their adult life (age - 14)
invalid_emp = df[df['employment_years'] > (df['age'] - 14)]
print('Records where employment_years > (age - 14):', len(invalid_emp))

df = df[df['employment_years'] <= (df['age'] - 14)]
print('Shape after employment validation:', df.shape)

### Step 14: Verify Income Values (No Negative or Zero Values)

In [ ]:
print('Records with income <= 0:', (df['income'] <= 0).sum())
print('Min income:', df['income'].min())
print('Max income:', df['income'].max())

### Step 15: Verify Loan Amount Values

In [ ]:
print('Records with loan_amount <= 0:', (df['loan_amount'] <= 0).sum())
print('Min loan amount:', df['loan_amount'].min())
print('Max loan amount:', df['loan_amount'].max())

### Step 16: Verify Loan-to-Income Ratio Range

In [ ]:
print('Records with loan_income_ratio > 1.0:', (df['loan_income_ratio'] > 1.0).sum())
print('Records with loan_income_ratio < 0:', (df['loan_income_ratio'] < 0).sum())
print('Min ratio:', df['loan_income_ratio'].min())
print('Max ratio:', df['loan_income_ratio'].max())

### Step 17: Standardize Categorical Strings (Trim Whitespace)

In [ ]:
cat_cols = ['home_ownership', 'loan_purpose', 'loan_grade', 'previous_default']
for col in cat_cols:
    df[col] = df[col].astype(str).str.strip().str.upper()

print('Categorical columns standardized successfully.')

### Step 18: Check Unique Categories in Categorical Features

In [ ]:
print('Home Ownership:', df['home_ownership'].unique())
print('Loan Purpose:', df['loan_purpose'].unique())
print('Loan Grade:', sorted(df['loan_grade'].unique()))
print('Previous Default:', df['previous_default'].unique())

### Step 19: Check Target Variable (Default) Counts and Proportions

In [ ]:
print('Default Counts:')
print(df['default'].value_counts())
print('\nDefault Percentage:')
print((df['default'].value_counts(normalize=True) * 100).round(2))

### Step 20: Final Overview of Cleaned Data

In [ ]:
print('Cleaned Data Shape:', df.shape)
print('Null values count:', df.isnull().sum().sum())
print('Duplicates count:', df.duplicated().sum())
df.describe().round(2)

### Step 21: Save Cleaned Dataset to CSV

In [ ]:
output_path = '../data/processed/cleaned_data.csv'
df.to_csv(output_path, index=False)
print(f'Cleaned dataset successfully saved to: {output_path}')